# Task 3 — log-target loss and tail check (v7)

The v5 log-MAE amount model improves all five development folds, but true spending ≥฿50,000 is still underpredicted on average. Test one prespecified change: fit the positive-only CatBoost on `log1p(target)` with **RMSE** instead of MAE. Keep all other components fixed. Compare the RMSE amount and a fixed 50/50 log-MAE/log-RMSE amount blend on the same five folds, then check the reused holdout once. Test and Kaggle scores are not used.

In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task3.models.spending_models import PositiveGate,PrefitBlend,WeightedPair
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split,StratifiedKFold
SRC=ROOT/'src';TASK=SRC/'task3';SAVE=TASK/'save';RES=TASK/'models'/'results'
DATA='eda_combined_v1';NAME='t3_log_rmse_amount_cv5_v7'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path'])
y=train.spending_30d.astype(float)
bands=pd.cut(y,bins=[-1,0,100,1000,10000,50000,np.inf],labels=False)
dev_idx,hold_idx=train_test_split(np.arange(len(y)),test_size=.20,stratify=bands,random_state=SEED)
dev=train.iloc[dev_idx].reset_index(drop=True);hold=train.iloc[hold_idx]
dev_bands=bands.iloc[dev_idx].reset_index(drop=True)
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev,dev_bands))
features=[c for c in dev if c not in ['player_id','spending_30d']]
old=joblib.load(SAVE/'t3_gate_blend_cv5_v2.joblib')
assert old.first_weight==.8 and old.first_model.threshold==.5 and len(features)==48
print(len(dev),'development rows',len(hold),'reused holdout rows')

83200 development rows 20800 reused holdout rows


## 1. Reconstruct the old and selected log-MAE OOF references

In [2]:
parts=[]
for i,(_,check) in enumerate(folds,1):
    gate=pd.read_parquet(RES/f'round3_gate_fold{i}_v1.parquet')
    blend=pd.read_parquet(RES/f'round3_blend_fold{i}_v1.parquet')
    assert gate.row_pos.tolist()==check.tolist()==blend.row_pos.tolist()
    assert gate.true.equals(blend.true)
    continuous=np.maximum(old.second_model.catboost_weight*blend.cat+
         (1-old.second_model.catboost_weight)*blend.lgb,0)
    baseline=.8*np.where(gate.probability>=.5,np.maximum(gate.positive_amount,0),0)+.2*continuous
    parts.append(pd.DataFrame({'fold':i,'row_pos':check,'true':gate.true,
         'probability':gate.probability,'continuous':continuous,'baseline':baseline}))
base=pd.concat(parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert np.array_equal(base.row_pos,np.arange(len(dev)))
assert np.array_equal(base.true,dev.spending_30d)
def fold_mae(pred):
    return np.array([mean_absolute_error(base.loc[base.fold==i,'true'],pred[base.fold==i]) for i in range(1,6)])
old_scores=fold_mae(base.baseline.to_numpy())
assert abs(old_scores.mean()-2895.92)<.2
print('Baseline fold MAE:',old_scores.round(2),'mean',old_scores.mean())
log_parts=[]
for i,(_,check) in enumerate(folds,1):
    part=pd.read_parquet(RES/f't3_log_positive_amount_cv5_v5-fold{i}.parquet')
    assert part.row_pos.tolist()==check.tolist()
    log_parts.append(part)
log_amount=pd.concat(log_parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert np.array_equal(log_amount.row_pos,np.arange(len(dev)))
log_pred=.8*np.where(base.probability>=.5,log_amount.amount,0)+.2*base.continuous
log_scores=fold_mae(log_pred.to_numpy())
assert abs(log_scores.mean()-2814.9680123616345)<.05
print('Selected log-MAE fold MAE:',log_scores.round(2),'mean',log_scores.mean())

Baseline fold MAE: [2815.58 2973.69 2797.07 2929.51 2963.76] mean 2895.921540742373
Selected log-MAE fold MAE: [2743.14 2886.62 2711.28 2848.55 2885.25] mean 2814.9680123616345


## 2. Fit log-RMSE positive amount, save, and evaluate

The same preprocessing pipeline is refitted on positive training rows inside each fold. This cell saves a distinct development-fitted model and evaluates it on the reused holdout; the five-fold scores are the primary comparison.

In [3]:
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
def amount_model():
    reg=clone(old.first_model.positive_amount_model)
    reg.set_params(regressor__loss_function='RMSE')
    return TransformedTargetRegressor(regressor=reg,func=np.log1p,
        inverse_func=np.expm1,check_inverse=False)
key={'dataset':DATA,'created_at':card['created_at'],'seed':SEED,
     'source':'t3_log_positive_amount_cv5_v5','change':'log1p positive amount RMSE loss'}
parts=[]
for i,(fit,check) in enumerate(folds,1):
    path=RES/f'{NAME}-fold{i}.parquet';meta=path.with_suffix('.json')
    if path.exists()!=meta.exists():raise RuntimeError(f'Incomplete fold {i}')
    if path.exists():
        assert json.loads(meta.read_text(encoding='utf-8'))=={'key':key,'fold':i}
        part=pd.read_parquet(path)
    else:
        if any(p.exists() for p in paths):raise RuntimeError('Saved model has missing OOF fold')
        positive=fit[dev.iloc[fit].spending_30d.to_numpy()>0]
        m=amount_model();m.fit(dev.iloc[positive][features],dev.iloc[positive].spending_30d)
        pred=np.maximum(np.asarray(m.predict(dev.iloc[check][features]),dtype=float),0)
        part=pd.DataFrame({'fold':i,'row_pos':check,'amount':pred})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({'key':key,'fold':i},indent=2),encoding='utf-8')
        print('Fold',i,'fitted',flush=True)
    assert np.array_equal(part.row_pos,check)
    parts.append(part)
rmse_amount=pd.concat(parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert np.array_equal(rmse_amount.row_pos,np.arange(len(dev)))
rmse_pred=.8*np.where(base.probability>=.5,rmse_amount.amount,0)+.2*base.continuous
rmse_scores=fold_mae(rmse_pred.to_numpy())
if not any(p.exists() for p in paths):
    positive=dev.spending_30d>0
    amount=amount_model();amount.fit(dev.loc[positive,features],dev.loc[positive,'spending_30d'])
    model=WeightedPair(PositiveGate(old.first_model.occurrence_model,amount,.5),old.second_model,.8)
    hold_pred=model.predict(hold[features])
    save_model(task=3,name=NAME,model=model,data_name=DATA,
        card={'idea':'Log1p positive amount with RMSE objective',
              'source_model':'t3_log_positive_amount_cv5_v5','seed':SEED,
              'development_cv_mean_MAE_thb':float(rmse_scores.mean())})
    evaluate_model(task=3,name=NAME,score=float(mean_absolute_error(hold.spending_30d,hold_pred)),
        metric_name='MAE_thb',higher_is_better=False,
        card={'split':'reused 20% holdout','development_fold_MAE_thb':rmse_scores.tolist(),
              'same_fold_log_MAE_thb':log_scores.tolist()})
else:
    assert all(p.exists() for p in paths)
    model=joblib.load(paths[0]);result=json.loads(paths[2].read_text(encoding='utf-8'))
    hold_pred=model.predict(hold[features])
    assert abs(result['score']-mean_absolute_error(hold.spending_30d,hold_pred))<.05
display(pd.DataFrame({'selected_log_MAE':log_scores,'log_RMSE':rmse_scores,
                      'delta':rmse_scores-log_scores},index=range(1,6)).round(2))
print('RMSE mean MAE:',rmse_scores.mean(),'reused holdout:',mean_absolute_error(hold.spending_30d,hold_pred))

,selected_log_MAE,log_RMSE,delta
1,2743.14,2763.02,19.88
2,2886.62,2910.93,24.31
3,2711.28,2734.90,23.62
4,2848.55,2858.01,9.46
5,2885.25,2900.37,15.12


RMSE mean MAE: 2833.446559853679 reused holdout: 2794.8357918637194


## 3. Fixed half blend of positive amount models

This uses the two independently fold-fitted OOF amount predictions. The blend weight was specified before seeing this round's result. For the saved development model, combine the already fitted v5 log-MAE amount model with the fitted RMSE amount model.

In [4]:
BLEND='t3_log_amount_halfblend_cv5_v7'
blend_paths=[SAVE/f'{BLEND}.joblib',SAVE/f'{BLEND}.json',SAVE/f'{BLEND}-evaluation.json']
blend_amount=.5*log_amount.amount.to_numpy()+.5*rmse_amount.amount.to_numpy()
blend_pred=.8*np.where(base.probability>=.5,blend_amount,0)+.2*base.continuous
blend_scores=fold_mae(blend_pred)
if not any(p.exists() for p in blend_paths):
    log_model=joblib.load(SAVE/'t3_log_positive_amount_cv5_v5.joblib')
    amount_blend=PrefitBlend(log_model.first_model.positive_amount_model,
                            model.first_model.positive_amount_model,.5)
    blend_model=WeightedPair(PositiveGate(old.first_model.occurrence_model,amount_blend,.5),
                             old.second_model,.8)
    blend_hold=blend_model.predict(hold[features])
    save_model(task=3,name=BLEND,model=blend_model,data_name=DATA,
        card={'idea':'Fixed half blend of log-MAE and log-RMSE positive amount models',
              'source_models':['t3_log_positive_amount_cv5_v5',NAME],
              'seed':SEED,'development_cv_mean_MAE_thb':float(blend_scores.mean())})
    evaluate_model(task=3,name=BLEND,score=float(mean_absolute_error(hold.spending_30d,blend_hold)),
        metric_name='MAE_thb',higher_is_better=False,
        card={'split':'reused 20% holdout','development_fold_MAE_thb':blend_scores.tolist(),
              'same_fold_log_MAE_thb':log_scores.tolist()})
else:
    assert all(p.exists() for p in blend_paths)
    blend_model=joblib.load(blend_paths[0]);blend_hold=blend_model.predict(hold[features])
    blend_result=json.loads(blend_paths[2].read_text(encoding='utf-8'))
    assert abs(blend_result['score']-mean_absolute_error(hold.spending_30d,blend_hold))<.05
display(pd.DataFrame({'selected_log_MAE':log_scores,'log_RMSE':rmse_scores,
                      'fixed_half_blend':blend_scores},index=range(1,6)).round(2))
print('Blend mean MAE:',blend_scores.mean(),'reused holdout:',mean_absolute_error(hold.spending_30d,blend_hold))

,selected_log_MAE,log_RMSE,fixed_half_blend
1,2743.14,2763.02,2737.92
2,2886.62,2910.93,2886.30
3,2711.28,2734.90,2712.05
4,2848.55,2858.01,2838.04
5,2885.25,2900.37,2881.50


Blend mean MAE: 2811.1626967914617 reused holdout: 2777.25918468434


## 4. OOF error tradeoff by spending group

The fixed blend may improve the rare high-spend tail while worsening the many zero-spend rows. These are descriptive slices of the same development OOF predictions, not new validation sets.

In [5]:
group_masks={'all':np.ones(len(base),dtype=bool),
             'zero':base.true.to_numpy()==0,
             'positive':base.true.to_numpy()>0,
             '10k_to_50k':(base.true.to_numpy()>=10000)&(base.true.to_numpy()<50000),
             '50k_plus':base.true.to_numpy()>=50000,
             'false_nonzero':(base.true.to_numpy()==0)&(base.probability.to_numpy()>=.5)}
rows=[]
for label,mask in group_masks.items():
    rows.append({'group':label,'rows':int(mask.sum()),
        'log_MAE':mean_absolute_error(base.true.to_numpy()[mask],log_pred.to_numpy()[mask]),
        'log_RMSE':mean_absolute_error(base.true.to_numpy()[mask],rmse_pred.to_numpy()[mask]),
        'fixed_half_blend':mean_absolute_error(base.true.to_numpy()[mask],blend_pred.to_numpy()[mask])})
display(pd.DataFrame(rows).set_index('group').round(2))

,rows,log_MAE,log_RMSE,fixed_half_blend
group,,,,
all,83200,2814.97,2833.45,2811.16
zero,40039,1923.42,1941.42,1932.42
positive,43161,3642.03,3660.94,3626.34
10k_to_50k,4470,4358.16,4753.31,4483.44
50k_plus,6786,18294.04,18146.97,18123.58
false_nonzero,10849,7080.77,7147.24,7114.00


## Decision

Prefer the existing v6 full-data model unless RMSE or its prespecified half blend gives a meaningful, consistent five-fold MAE reduction. Report holdout only as a reused secondary check and avoid a weight/threshold grid on these same rows.